In [ ]:
pip install findspark

In [ ]:
pip install findspark

In [ ]:
pip install confluent-kafka

In [ ]:
pip install matplotlib seaborn pandas kafka-python wordcloud pyspark findspark neo4j

In [2]:
from kafka import KafkaProducer
import json
import time
from neo4j import GraphDatabase
import threading
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
import re

# Initialize Spark session
import findspark
findspark.init()

spark = SparkSession.builder \
    .appName("LexiconSearchStream") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:3.5.3") \
    .getOrCreate()

# Initialize Kafka producer
class KafkaPublisher:
    def __init__(self, kafka_server, topic):
        self.producer = KafkaProducer(
            bootstrap_servers=kafka_server,
            value_serializer=lambda v: json.dumps(v).encode('utf-8')
        )
        self.topic = topic

    def publish(self, word_data):
        # Prepare the message with timestamp and word data
        message = {
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "searched_word": word_data["word"],
            "part_of_speech": word_data["part_of_speech"],
            "sentiment_label": word_data["sentiment_label"],
            "derived_words": word_data["DerivedWord"],
            "synonyms": word_data["synonyms"],
            "antonyms": word_data["antonyms"],
            "context": word_data["context"]
        }
        
        # Send the message to Kafka topic
        self.producer.send(self.topic, message)
        print(f"Published to Kafka: {message}")

    def close(self):
        self.producer.close()

# Initialize Neo4j connection
class LexiconSearch:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        if self.driver:
            self.driver.close()

    def search_word(self, word):
        query = """
        MATCH (w:Word {name: $word})
        OPTIONAL MATCH (w)-[:HAS_DERIVED_WORD]->(drw:DerivedWord)
        OPTIONAL MATCH (w)-[:HAS_SYNONYM]->(syn:Synonym)
        OPTIONAL MATCH (w)-[:HAS_ANTONYM]->(ant:Antonym)
        OPTIONAL MATCH (w)-[:HAS_CONTEXT]->(ctx:Context)
        RETURN w.name AS word, 
               w.part_of_speech AS part_of_speech, 
               w.sentimental_label AS sentiment_label, 
               collect(DISTINCT drw.name) AS DerivedWord, 
               collect(DISTINCT syn.name) AS synonyms, 
               collect(DISTINCT ant.name) AS antonyms, 
               collect(DISTINCT ctx.description) AS context
        """
        with self.driver.session() as session:
            result = session.run(query, word=word)
            return result.single()

# Define schema for the incoming data from Kafka
schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("searched_word", StringType(), True),
    StructField("part_of_speech", StringType(), True),
    StructField("sentiment_label", StringType(), True),
    StructField("derived_words", ArrayType(StringType()), True),
    StructField("synonyms", ArrayType(StringType()), True),
    StructField("antonyms", ArrayType(StringType()), True),
    StructField("context", ArrayType(StringType()), True)
])

# Kafka Streaming Logic with Real-Time Word Frequency, Sentiment Frequency and Combined Analysis
def start_kafka_stream_with_analysis():
    # Read data stream from Kafka topic
    kafka_stream = spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", "localhost:9092") \
        .option("subscribe", "lexicon_searches") \
        .load()

    # Parse the message (assumes JSON format)
    search_data = kafka_stream.select(from_json(col("value").cast("string"), schema).alias("data"))

    # Extract the fields
    search_data = search_data.select("data.timestamp", "data.searched_word", "data.sentiment_label")

    # Word Frequency by searched_word
    word_frequency = search_data.groupBy("searched_word").count()

    # Sentiment Frequency by sentiment_label
    sentiment_frequency = search_data.groupBy("sentiment_label").count()

    # Combination of searched_word and sentiment_label with counts
    word_sentiment_combination = search_data.groupBy("searched_word", "sentiment_label").count()

    # Write the word frequency results to the console in the desired format
    word_frequency_query = word_frequency.writeStream \
        .format("console") \
        .outputMode("complete") \
        .option("truncate", "false") \
        .start()

    # Write the sentiment frequency results to the console in the desired format
    sentiment_frequency_query = sentiment_frequency.writeStream \
        .format("console") \
        .outputMode("complete") \
        .option("truncate", "false") \
        .start()

    # Write the combination results to the console in the desired format
    word_sentiment_combination_query = word_sentiment_combination.writeStream \
        .format("console") \
        .outputMode("complete") \
        .option("truncate", "false") \
        .start()

    # Await termination
    word_frequency_query.awaitTermination()
    sentiment_frequency_query.awaitTermination()
    word_sentiment_combination_query.awaitTermination()

# Transformation Function: Lowercase and remove punctuation
def transform_word(word):
    # Convert to lowercase
    word = word.lower()
    
    # Remove punctuation and spaces (example)
    word = re.sub(r'[^\w\s]', '', word)
    
    return word

# User Input Logic with Max Inputs Limit
def user_input():
    # Prompt the user to define the number of inputs
    try:
        max_inputs = int(input("How many words would you like to input? ").strip())
        if max_inputs <= 0:
            print("Please enter a positive number.")
            return
    except ValueError:
        print("Invalid input. Please enter a valid number.")
        return
    lexicon = LexiconSearch("neo4j+s://6428c331.databases.neo4j.io", "neo4j", "ExAcRcLBx7rZPXUVxGCo0Aqbpu8_N3ZIu7awx_yOTWQ")

    #lexicon = LexiconSearch("neo4j+s://59784ccc.databases.neo4j.io", "neo4j", "MedxoBPypewRCwgUyr1KVnOduts5S25YPD61GViKx7Y")
    kafka_publisher = KafkaPublisher("localhost:9092", "lexicon_searches")
    
    inputs_count = 0
    while inputs_count < max_inputs:
        user_word = input(f"Enter a word to search ({inputs_count + 1}/{max_inputs}): ").strip()

        if not user_word:
            print("No word entered. Exiting.")
            break

        # Apply transformation (e.g., change word to lowercase and remove punctuation)
        transformed_word = transform_word(user_word)

        # Fetch word data from Neo4j for the transformed word
        word_data = lexicon.search_word(transformed_word)

        if word_data:
            # Publish the word data to Kafka
            kafka_publisher.publish(word_data)
        else:
            print(f"No data found for the word '{transformed_word}'.")

        inputs_count += 1
    
    print("Max input reached or no input provided. Exiting.")
    lexicon.close()
    kafka_publisher.close()

# Running both Kafka stream and user input in parallel using threads
if __name__ == "__main__":
    # Start Spark streaming in a separate thread
    streaming_thread = threading.Thread(target=start_kafka_stream_with_analysis)
    streaming_thread.start()

    # Run the user input logic in the main thread
    user_input()  # Let the user define the number of inputs


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:417)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1864)
	at org.apache.spark.SparkContext.$anonfun$new$16(SparkContext.scala:545)
	at org.apache.spark.SparkContext.$anonfun$new$16$adapted(SparkContext.scala:545)
	at scala.collection.immutable.List.foreach(List.scala:334)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:545)
	at org.apache.spar

In [3]:
import os
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))
print("hadoop.home.dir:", os.environ.get("hadoop.home.dir"))


HADOOP_HOME: None
hadoop.home.dir: None
